In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Atur level optimasi transpiler

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Perangkat kuantum nyata rentan terhadap noise dan error gate, sehingga mengoptimalkan sirkuit untuk mengurangi kedalaman dan jumlah gate-nya bisa secara signifikan meningkatkan hasil yang diperoleh dari eksekusi sirkuit tersebut.
Fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) memiliki satu argumen posisional yang wajib, `optimization_level`, yang mengontrol seberapa besar usaha yang dikeluarkan transpiler untuk mengoptimalkan sirkuit. Argumen ini bisa berupa integer dengan nilai 0, 1, 2, atau 3.
Level optimasi yang lebih tinggi menghasilkan sirkuit yang lebih optimal dengan konsekuensi waktu kompilasi yang lebih lama.
Tabel berikut menjelaskan optimasi yang dilakukan pada setiap pengaturan.

<table>
  <thead>
    <tr>
      <th>Level Optimasi</th>
      <th>Deskripsi</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Tanpa optimasi: biasanya digunakan untuk karakterisasi hardware
        - Translasi dasar
        - Layout/Routing: `TrivialLayout`, di mana ia memilih nomor qubit fisik yang sama dengan virtual dan menyisipkan SWAP agar berfungsi (menggunakan `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Optimasi ringan:
        -   Layout/Routing: Layout pertama kali dicoba dengan `TrivialLayout`. Jika SWAP tambahan diperlukan, layout dengan jumlah SWAP minimum ditemukan menggunakan `SabreSwap`, lalu menggunakan `VF2LayoutPostLayout` untuk mencoba memilih qubit terbaik dalam grafik.
        -   `InverseCancellation`
        -   Optimasi gate 1Q
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Optimasi menengah:
          - Layout/Routing: Level optimasi 1 (tanpa trivial) + heuristik yang dioptimalkan dengan kedalaman pencarian dan percobaan fungsi optimasi yang lebih besar. Karena `TrivialLayout` tidak digunakan, tidak ada upaya untuk menggunakan nomor qubit fisik dan virtual yang sama.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Optimasi tinggi:
        - Level optimasi 2 + heuristik yang dioptimalkan lebih lanjut pada layout/routing dengan usaha/percobaan yang lebih besar
        - Sintesis ulang blok dua-qubit menggunakan [Dekomposisi KAK Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Pass yang memutus unitaritas:
          * `OptimizeSwapBeforeMeasure`: Memindahkan pengukuran untuk menghindari SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: Menghapus gate sebelum pengukuran yang tidak akan memengaruhi hasil pengukuran
      </td>
    </tr>
  </tbody>
</table>

## Level optimasi dalam aksi
Karena gate dua-qubit biasanya menjadi sumber error yang paling signifikan, kita bisa mengukur "efisiensi hardware" dari transpilasi secara kasar dengan menghitung jumlah gate dua-qubit dalam sirkuit yang dihasilkan.
Di sini, kita akan mencoba berbagai level optimasi pada sirkuit input yang terdiri dari unitary acak diikuti oleh gate SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Kita akan menggunakan backend tiruan `FakeSherbrooke` dalam contoh-contoh ini. Pertama, mari transpilasi menggunakan level optimasi 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Sirkuit hasil transpilasi memiliki enam gate ECR dua-qubit.

Ulangi untuk level optimasi 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Sirkuit hasil transpilasi masih memiliki enam gate ECR, tapi jumlah gate single-qubit sudah berkurang.

Ulangi untuk level optimasi 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Ini menghasilkan hasil yang sama dengan level optimasi 1. Perlu dicatat bahwa meningkatkan level optimasi tidak selalu membuat perbedaan.

Ulangi lagi, dengan level optimasi 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Sekarang, hanya ada tiga gate ECR. Kita mendapatkan hasil ini karena pada level optimasi 3, Qiskit mencoba mensintesis ulang blok gate dua-qubit, dan gate dua-qubit manapun bisa diimplementasikan menggunakan paling banyak tiga gate ECR. Kita bisa mendapatkan lebih sedikit gate ECR lagi jika kita mengatur `approximation_degree` ke nilai kurang dari 1, yang memungkinkan transpiler membuat aproksimasi yang mungkin menimbulkan sedikit error dalam dekomposisi gate (lihat [Parameter yang umum digunakan untuk transpilasi](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Sirkuit ini hanya memiliki dua gate ECR, tapi ini adalah sirkuit aproksimasi. Untuk memahami seberapa berbeda efeknya dari sirkuit eksak, kita bisa menghitung fidelitas antara operator unitary yang diimplementasikan sirkuit ini dengan unitary eksak. Sebelum melakukan komputasi, kita terlebih dahulu mereduksi sirkuit hasil transpilasi yang berisi 127 qubit menjadi sirkuit yang hanya berisi qubit aktif, yaitu dua qubit.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>